# Model A
---

Load the preprocessed datasets, vocabulary, config and dataloaders using the following cell.

In [1]:
from utils import load_sets, load_bpe, load_config, get_dataloaders

train_set, val_set, test_set = load_sets()
config = load_config()

# Load BPE tokenizer
bpe = load_bpe(config.get('BPE_DIR', './data/bpe'))

train_loader, val_loader, test_loader = get_dataloaders(
    train_set, val_set, test_set,
    vocab={},  # keep empty for BPE mode
    max_length=config['MAX_LENGTH'],
    batch_size=config['BATCH_SIZE'],
    bpe=bpe
)

# Model/token IDs (important!)
INPUT_DIM = bpe.get_vocab_size()
OUTPUT_DIM = bpe.get_vocab_size()

PAD_IDX = bpe.pad_id
SOS_IDX = bpe.sos_id
EOS_IDX = bpe.eos_id
UNK_IDX = bpe.unk_id

In [2]:
train_set.head()

,input,response
0,<user> lsof | <cmd> <path>,<user> thanks. :d
1,<user> '/server irc.undernet.org' <user> .org,<user> have you try this already? it seems its...
2,<user> is there a command like halt or reboot ...,<user> suspend?
3,<user> ati card?,"<user> yeah, i'll go to other channel now"
4,<user> cdrtools which the gui apps use to burn,<user> - thanks


In [3]:
config

{'TOKENIZER_MODE': 'bpe',
 'SPECIAL_TOKENS': ['<PAD>',
  '<SOS>',
  '<EOS>',
  '<UNK>',
  '<SEP>',
  '<S0>',
  '<S1>',
  '<url>',
  '<path>',
  '<ip>',
  '<user>',
  '<version>',
  '<email>',
  '<cmd>'],
 'MAX_LENGTH': 22,
 'MIN_LENGTH': 2,
 'BATCH_SIZE': 64,
 'TRAIN_SIZE': 226449,
 'VAL_SIZE': 25161,
 'TEST_SIZE': 27957,
 'BPE_DIR': './data/bpe',
 'BPE_VOCAB_SIZE': 16000,
 'BPE_TARGET_VOCAB_SIZE': 16000,
 'BPE_MIN_FREQ': 2}

In [4]:
import torch

# Always use cuda if available (NVIDIA GPU), then try for apple silicon, otherwise just a plain old CPU (but training will be vastly slower)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f'Using device: {torch.cuda.get_device_name(0)}')
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print('Using device: MPS (Apple Silicon)')
else:
    device = torch.device("cpu")
    print('Using device: CPU')

Using device: MPS (Apple Silicon)


Quick plan:

1. Define the Encoder  — GRU that reads input, returns ~~all hidden states~~ + final hidden state
2. Define the Decoder  — GRU that generates output token by token using final encoder hidden state
3. Define the Seq2Seq  — wrapper that connects encoder and decoder, handles teacher forcing
4. Training loop       — with validation loss monitoring
5. Inference function  — greedy decoding for generating responses
6. Evaluation          — BLEU score on test set + manual examples

## Encoder

---

The encoder will read the input sequences we've prepared, and will output a single final hidden state to initialise the decoder. So, the architecture is actually quite simple:
- An embedding layer (dataset has low GloVe coverage, so we can train embeddings from scratch).
- A stack of Gated Recurrent Units (combats vanishing gradients, fewer parameters to train than an LSTM, with comparable performance).
- Dropout as a regularisation technique to combat overfitting and improve generalisation (only applied during training). We'll apply dropout between the embedding layer, and between the layers in the GRU.

Defining a sequential model in PyTorch follows the usual pattern:
- Extend the [`nn.Module`](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html) base class.
- Initialise your layers in the constructor.
- Override the `forward` method, explicitly feeding X inputs through your layers.

Please read the [GRU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GRU.html) documentation, it explains a lot! :)

In [5]:
import torch
import torch.nn as nn

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_dim, num_layers, dropout_proba):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, 
            embed_dim, 
            padding_idx=padding_idx # Pad out to sequence length
        )
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim, # The dimension of the hidden state vector maintained through timesteps
            num_layers=num_layers,
            batch_first=True, # This just moves batch to the first dimension, to match our dataloader implementation
            dropout=dropout_proba if num_layers > 1 else 0 # also apply dropout between GRU layers
        )
        self.dropout = nn.Dropout(dropout_proba)

    def forward(self, X):
        # X: [batch_size, seq_length]
        X = self.embedding(X)
        X = self.dropout(X) # Applying dropout here is standard for seq2seq, it's the largest parameter space in the network
        # X: [batch_size, seq_length, embed_dim]
        _, hidden = self.gru(X)
        # hidden: [num_layers, batch_size, hidden_dim] (our final hidden state)
        
        # We'll ignore the outputs for now, as this model won't use attention
        return hidden

## Decoder

---

The decoder must first initialise its hidden state with the final output hidden state of the encoder. Then, at each timestep (**a forward pass through the network processing one token**):
1. Obtain the embedding for the previous token
2. Pass it through the GRU
3. Project the output to the vocabulary size to predict the next token (using a simple fully connected linear layer)

Outside of this, its layer initialisation is very similar to the encoder. 

Note we're also training separate embeddings in the decoder rather using shared embeddings between the encoder and decoder (called _weight tying_ -- possible note in the report). This is to keep the embeddings specialised as the encoder and decoder objectives are fundamentally different: the encoder must encode meaning into the hidden state, the decoder must utilise this state to predict the next token given the previous one. Therefore, allowing both to represent tokens in a way that optimises their objectives is the intent.

In [6]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, padding_idx, hidden_dim, num_layers, dropout_proba):
        super().__init__()
        self.embedding = nn.Embedding(
            vocab_size, 
            embed_dim, 
            padding_idx=padding_idx
        )
        self.gru = nn.GRU(
            embed_dim,
            hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_proba if num_layers > 1 else 0
        )
        self.linear = nn.Linear(hidden_dim, vocab_size) # Simply project from the hidden dimension size down to the vocabulary size 
        self.dropout = nn.Dropout(dropout_proba)

    def forward(self, X, hidden):
        # X: [batch_size] (per sample, just one token)
        X = X.unsqueeze(1)
        # X: [batch_size, 1] (add seq_length dimension, which is just 1, since we're only processing one token at a time)
        X = self.embedding(X)
        X = self.dropout(X)
        # X: [batch_size, 1, embed_dim]
        output, hidden = self.gru(X, hidden)
        # output: [batch_size, 1, hidden_dim] (?)
        # hidden: [num_layers, batch_size, hidden_dim] (the next hidden state to use at the next timestep)

        pred = self.linear(output.squeeze(1)) # Convert back to [batch_size] and project to vocabulary
        # pred: [batch_size, vocab_size] (this is our next token prediction!)

        return pred, hidden

## Seq2Seq Wrapper

---

This is the wrapper model that will be built from the encoder and decoder. It will:
1. Pass the full input sequence through the encoder to produce a final hidden state
2. Use this final hidden state to initialise the decoder
3. Loop through each decoding timestep and feed each token in one at a time
4. Apply **teacher forcing** during training only (feeding the ground truth labels in at each timestep to improve stability and reduce _exposure bias_I)

The variables passed into the `forward` method are those we prepared at the data preparation stage:
- `encoder_input`: the encoded input sequence (for the encoder)
- `decoder_input`: the encoded response with SOS prepended

In [7]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, encoder_input, decoder_input, teacher_forcing_proba=0.5):
        # encoder_input: [batch_size, encoder_input_length]
        # decoder_input: [batch_size, decoder_input_length]
        
        batch_size = encoder_input.shape[0]
        decoder_input_length = decoder_input.shape[1]
        vocab_size = self.decoder.linear.out_features

        # A tensor for storing the decoder predictions (ensure tensor is on the same device as the model)
        outputs = torch.zeros(batch_size, decoder_input_length, vocab_size).to(self.device)

        # Forward pass through the encoder to obtain the final hidden state
        hidden = self.encoder(encoder_input)

        # First decoder input is the <SOS> token for every sample
        next_decoder_input = decoder_input[:, 0]

        # Every timestep: every token in the sequence, produce a prediction
        for timestep in range(1, decoder_input_length):
            pred, hidden = self.decoder(next_decoder_input, hidden) # Use the next decoder input and last hidden state to produce the next token prediction
            outputs[:, timestep, :] = pred # For all samples (tokens) in the batch, assign all decoder predictions (probability distributions) at this timestep

            # Teacher forcing (using the ground truth or model prediction with highest probability)
            # We'll apply teacher forcing with the given probability. This probability will balance learning 
            # from ground truth and learning from experience with the end goal of improving ability to generalise on unseen data (during inference)
            use_teacher_forcing = torch.rand(1).item() < teacher_forcing_proba
            next_decoder_input = decoder_input[:, timestep] if use_teacher_forcing else pred.argmax(1) # ground truth OR best prediction

        return outputs

## Training Loop

---

This is the main training loop, where we'll use cross entropy loss, Adam optimiser, and gradient clipping to manage exploding gradients (this is common in BPTT).

We can use a `max_norm=1.0` to scale the gradient vector down if its norm exceeds 1 to help keep the weight updates stable.

First things first: let's define the training and model hyperparameters, and instantiate the model.

In [9]:
# Training Parameters
EPOCHS = 10
CLIP_MAX_NORM = 1.0
TEACHER_FORCING_PROBA = 0.5

# Model Hyperparameters
ENCODER_EMBED_DIM = 128
ENCODER_HIDDEN_DIM = 256
ENCODER_NUM_LAYERS = 2
ENCODER_DROPOUT_PROBA = 0.3

DECODER_EMBED_DIM = 128
DECODER_HIDDEN_DIM = 256
DECODER_NUM_LAYERS = 2
DECODER_DROPOUT_PROBA = 0.3

# from previous cell:
# INPUT_DIM = bpe.get_vocab_size()
# OUTPUT_DIM = bpe.get_vocab_size()
# PAD_IDX = bpe.pad_id
# SOS_IDX = bpe.sos_id
# EOS_IDX = bpe.eos_id
# UNK_IDX = bpe.unk_id

model_a = Seq2Seq(
    encoder=Encoder(
        vocab_size=INPUT_DIM,
        embed_dim=ENCODER_EMBED_DIM,
        padding_idx=PAD_IDX,
        hidden_dim=ENCODER_HIDDEN_DIM,
        num_layers=ENCODER_NUM_LAYERS,
        dropout_proba=ENCODER_DROPOUT_PROBA
    ),
    decoder=Decoder(
        vocab_size=OUTPUT_DIM,
        embed_dim=DECODER_EMBED_DIM,
        padding_idx=PAD_IDX,
        hidden_dim=DECODER_HIDDEN_DIM,
        num_layers=DECODER_NUM_LAYERS,
        dropout_proba=DECODER_DROPOUT_PROBA
    ),
    device=device
).to(device)  # Move model to device

In [14]:
#!mkdir -p models/model_a
from pathlib import Path

Path("models/model_a_bpe").mkdir(parents=True, exist_ok=True)

In [15]:
from torch.optim import Adam
from pathlib import Path
import torch
import torch.nn as nn
import time
import re

# Assumes these already exist from previous cells:
# model_a, train_loader, val_loader, device
# EPOCHS, CLIP_MAX_NORM, TEACHER_FORCING_PROBA
# bpe, PAD_IDX, INPUT_DIM, OUTPUT_DIM

checkpoint_dir = Path("models/model_a_bpe")
checkpoint_dir.mkdir(parents=True, exist_ok=True)

model_a_baseline_state_dict = checkpoint_dir / "model_a_baseline.pt"

# BPE-aware loss
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
optimiser = Adam(model_a.parameters(), lr=1e-3)

# Utility: sort checkpoints by epoch number


def ckpt_epoch(path: Path):
    m = re.search(r"checkpoint_epoch_(\d+)\.pt$", path.name)
    return int(m.group(1)) if m else -1


existing_checkpoints = sorted(checkpoint_dir.glob(
    "checkpoint_epoch_*.pt"), key=ckpt_epoch)

# Resume / load logic
if model_a_baseline_state_dict.exists():
    print("Loading baseline model weights...")
    state = torch.load(model_a_baseline_state_dict,
                       map_location=device, weights_only=True)
    model_a.load_state_dict(state)
    start_epoch = EPOCHS  # already finished baseline
elif existing_checkpoints:
    latest = existing_checkpoints[-1]
    print(f"Loading checkpoint: {latest.name}")
    checkpoint = torch.load(latest, map_location=device, weights_only=True)

    model_a.load_state_dict(checkpoint["model_state_dict"])
    optimiser.load_state_dict(checkpoint["optimiser_state_dict"])

    # if epoch saved, use it; otherwise infer from filename
    start_epoch = checkpoint.get("epoch", ckpt_epoch(latest))
    print(f"Resuming from epoch {start_epoch}/{EPOCHS}")
else:
    print("No checkpoints, training model from scratch...")
    start_epoch = 0


def train_epoch(model, loader, optimiser, criterion, device, clip_max_norm, teacher_forcing_proba):
    model.train()
    total_loss = 0.0
    total_batches = len(loader)
    log_interval = max(1, int(total_batches * 0.05))  # every 5%

    for batch_idx, (encoder_input_batch, decoder_input_batch, decoder_target_batch) in enumerate(loader):
        encoder_input_batch = encoder_input_batch.to(device)
        decoder_input_batch = decoder_input_batch.to(device)
        decoder_target_batch = decoder_target_batch.to(device)

        optimiser.zero_grad()

        predictions = model(
            encoder_input_batch,
            decoder_input_batch,
            teacher_forcing_proba=teacher_forcing_proba
        )  # [B, T, V]
        # [B, V, T] for CrossEntropyLoss
        predictions = predictions.permute(0, 2, 1)

        loss = criterion(predictions, decoder_target_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip_max_norm)
        optimiser.step()

        total_loss += loss.item()

        if (batch_idx + 1) % log_interval == 0:
            avg_loss = total_loss / (batch_idx + 1)
            progress = 100 * (batch_idx + 1) / total_batches
            print(
                f"  [{progress:.0f}%] Batch {batch_idx+1}/{total_batches} | Avg Loss: {avg_loss:.4f}")

    return total_loss / max(1, len(loader))


def val_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for encoder_input_batch, decoder_input_batch, decoder_target_batch in loader:
            encoder_input_batch = encoder_input_batch.to(device)
            decoder_input_batch = decoder_input_batch.to(device)
            decoder_target_batch = decoder_target_batch.to(device)

            predictions = model(
                encoder_input_batch,
                decoder_input_batch,
                teacher_forcing_proba=0.0
            )
            predictions = predictions.permute(0, 2, 1)

            loss = criterion(predictions, decoder_target_batch)
            total_loss += loss.item()

    return total_loss / max(1, len(loader))


# Main training loop
epoch_times = []

for epoch in range(start_epoch, EPOCHS):
    start = time.time()

    train_loss = train_epoch(
        model_a,
        train_loader,
        optimiser,
        criterion,
        device,
        CLIP_MAX_NORM,
        TEACHER_FORCING_PROBA
    )
    val_loss = val_epoch(
        model_a,
        val_loader,
        criterion,
        device
    )

    elapsed = time.time() - start
    epoch_times.append(elapsed)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Time: {elapsed:.2f}s"
    )

    # Save checkpoint (resume training)
    torch.save({
        "epoch": epoch + 1,
        "model_state_dict": model_a.state_dict(),
        "optimiser_state_dict": optimiser.state_dict(),
        "train_loss": train_loss,
        "val_loss": val_loss,
        "tokenizer_mode": "bpe",
        "bpe_vocab_size": bpe.get_vocab_size(),
        "pad_idx": PAD_IDX,
    }, checkpoint_dir / f"checkpoint_epoch_{epoch+1}.pt")

# Save model for inference (recommended: state_dict only)
torch.save(model_a.state_dict(), model_a_baseline_state_dict)

# Benchmarks
if epoch_times:
    print("\n--- Benchmark Summary ---")
    print(f"Device: {device}")
    print(f"Total training time: {sum(epoch_times):.2f}s")
    print(f"Average epoch time: {sum(epoch_times)/len(epoch_times):.2f}s")
    print(f"Fastest epoch: {min(epoch_times):.2f}s")
    print(f"Slowest epoch: {max(epoch_times):.2f}s")
else:
    print("Training already complete!")

No checkpoints, training model from scratch...
  [5%] Batch 176/3539 | Avg Loss: 6.7026
  [10%] Batch 352/3539 | Avg Loss: 6.5516
  [15%] Batch 528/3539 | Avg Loss: 6.4758
  [20%] Batch 704/3539 | Avg Loss: 6.4255
  [25%] Batch 880/3539 | Avg Loss: 6.3887
  [30%] Batch 1056/3539 | Avg Loss: 6.3607
  [35%] Batch 1232/3539 | Avg Loss: 6.3384
  [40%] Batch 1408/3539 | Avg Loss: 6.3190
  [45%] Batch 1584/3539 | Avg Loss: 6.3019
  [50%] Batch 1760/3539 | Avg Loss: 6.2873
  [55%] Batch 1936/3539 | Avg Loss: 6.2744
  [60%] Batch 2112/3539 | Avg Loss: 6.2630
  [65%] Batch 2288/3539 | Avg Loss: 6.2533
  [70%] Batch 2464/3539 | Avg Loss: 6.2431
  [75%] Batch 2640/3539 | Avg Loss: 6.2342
  [80%] Batch 2816/3539 | Avg Loss: 6.2262
  [85%] Batch 2992/3539 | Avg Loss: 6.2168
  [90%] Batch 3168/3539 | Avg Loss: 6.2097
  [94%] Batch 3344/3539 | Avg Loss: 6.2021
  [99%] Batch 3520/3539 | Avg Loss: 6.1954
Epoch 1/10 | Train Loss: 6.1946 | Val Loss: 6.2363 | Time: 2113.80s
  [5%] Batch 176/3539 | Avg Los

## Model Inference

---

In [26]:
import torch
from utils import clean


def _encode_input_bpe(text, bpe, max_input_len):
    text = clean(text)

    # Your tokenizer API
    if hasattr(bpe, "encode_sequence"):
        try:
            # if method expects max_len
            ids = bpe.encode_sequence(text, max_input_len)
        except TypeError:
            ids = bpe.encode_sequence(text)                 # if method doesn't
    else:
        raise AttributeError("BPETokenizer missing encode_sequence")

    # Ensure Python list[int]
    if torch.is_tensor(ids):
        ids = ids.tolist()
    ids = [int(x) for x in ids]

    # Safety pad/truncate (in case encode_sequence didn't enforce length)
    ids = ids[:max_input_len]
    if len(ids) < max_input_len:
        ids += [PAD_IDX] * (max_input_len - len(ids))

    return ids


def _decode_output_bpe(ids, bpe):
    # Try common decode names first
    for name in ["decode_sequence", "decode_ids", "decode"]:
        if hasattr(bpe, name):
            fn = getattr(bpe, name)
            try:
                return fn(ids)
            except TypeError:
                pass

    # Fallback: try id->token mapping attributes
    for name in ["id_to_token", "idx_to_token", "itos", "vocab_reversed"]:
        if hasattr(bpe, name):
            mapping = getattr(bpe, name)
            if callable(mapping):  # e.g. method
                return " ".join(str(mapping(i)) for i in ids)
            if isinstance(mapping, dict):
                return " ".join(str(mapping.get(i, "<UNK>") if not isinstance(next(iter(mapping.keys())), str)
                                else mapping.get(str(i), "<UNK>")) for i in ids)

    # Last resort: print IDs
    return " ".join(map(str, ids))


def generate_response_bpe(model, text, bpe, device, max_input_len=22, max_output_len=30):
    model.eval()

    input_ids = _encode_input_bpe(text, bpe, max_input_len)
    encoder_input = torch.tensor(
        input_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1, T]

    with torch.no_grad():
        hidden = model.encoder(encoder_input)
        next_token = torch.tensor([SOS_IDX], dtype=torch.long, device=device)

        generated_ids = []
        for _ in range(max_output_len):
            pred, hidden = model.decoder(next_token, hidden)
            next_token = pred.argmax(dim=-1)  # greedy decoding

            tid = int(next_token.item())
            if tid in (EOS_IDX, PAD_IDX):
                break
            generated_ids.append(tid)

    return _decode_output_bpe(generated_ids, bpe)

In [27]:
# Test
test_inputs = ["How do I install python3?"]
for text in test_inputs:
    response = generate_response_bpe(
        model=model_a,
        text=text,
        bpe=bpe,
        device=device,
        max_input_len=config["MAX_LENGTH"],
        max_output_len=30
    )
    print(f"Input:    {text}")
    print(f"Response: {response}\n")

Input:    How do I install python3?
Response: install install install

